# 2. VQ-VAE Training (Stage 1) - Standard Model

**Objective:** Train a Vector Quantized Variational Autoencoder (VQ-VAE) to represent mathematical sequences as discrete latent tokens.

This stage focuses on learning a robust codebook that can compress the "Prompt + Chain-of-Thought + Solution" (PCS) sequences. The resulting encoder and codebook will be used in the next stage to create the assorted dataset for LLM fine-tuning.

## 2.1 Environment Setup and Repository Cloning

To ensure reproducibility, this section automates the setup of the working environment:
1. **Google Drive Integration:** Mounts your personal Drive to store persistent data (checkpoints and processed datasets).
2. **Project Structure:** Automatically creates a `DLAI` folder in your Drive.
3. **Dependency Management:** Installs the `uv` package manager and resolves all requirements defined in `pyproject.toml`.
4. **Source Code:** Clones the `llama` branch from our GitHub repository to provide access to the `src` module and configuration files.

**Note for Evaluators:** Please authorize the Google Drive mount when prompted to allow the notebook to save and retrieve project files.

In [ ]:
import os, sys

# 1. Mount Google Drive
# Evaluators will need to accept the pop-up to connect their Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Setup directories on Drive
# Create the DLAI folder if it doesn't exist on their Drive
DRIVE_PROJECT_PATH = "/content/drive/MyDrive/DLAI"
if not os.path.exists(DRIVE_PROJECT_PATH):
    os.makedirs(DRIVE_PROJECT_PATH, exist_ok=True)
    print(f"Created project folder at: {DRIVE_PROJECT_PATH}")

# 3. UV Installation
# We use UV for much faster dependency management than standard pip
!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ['PATH'] = f"{os.path.expanduser('~')}/.cargo/bin:" + os.environ['PATH']

# 4. Clone the Repository (Branch: llama)
# If the local folder doesn't exist, clone the specific branch
%cd /content
if not os.path.exists("DLAI"):
    !git clone --branch llama https://github.com/irene-30/DLAI.git
else:
    print("Repo already exists, pulling latest changes...")
    !git -C DLAI pull

# 5. Synchronize pyproject.toml
# Copy the pyproject.toml from the cloned repo to the Drive folder (if necessary)
# or vice versa, to ensure that UV reads the correct dependencies.
!cp /content/DLAI/pyproject.toml {DRIVE_PROJECT_PATH}/pyproject.toml

# 6. Install dependencies via pyproject.toml
# This command reads the .toml file and installs everything necessary
%cd /content/DLAI
!uv pip install -e . --system

# 7. Add to the system path to allow imports from 'src'
sys.path.append("/content/DLAI")
%cd /content

print("✅ Setup completed successfully!")

## 2.2 Model Configuration and Tokenizer Initialization

We initialize the Llama 3.2 tokenizer and set up the experimental paths. We also define the main hyperparameters for the VQ-VAE training, such as batch size, learning rate, and the commitment cost which balances the reconstruction quality with codebook stability.

In [ ]:
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from datasets import load_dataset

# Internal module imports
from src.utils import get_llm_tokenizer, VQ_CODEBOOK_SIZE
from src.model.vae import VQVAEModel

# Device Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# --- PATH CONFIGURATION ---
# Output folder for model checkpoints
RESULTS_BASE_PATH = "/content/drive/My Drive/DLAI/experiments/vqvae_standard"
CHECKPOINT_DIR = os.path.join(RESULTS_BASE_PATH, "checkpoints")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Initialize the LLM tokenizer (Llama 3.2)
tokenizer = get_llm_tokenizer()
vocab_size = len(tokenizer)


# --- HYPERPARAMETERS ---
MAX_SEQ_LEN = 1024
BATCH_SIZE = 32
EPOCHS = 1
LR = 1e-4


## 2.3 Dataset Loading and Model Initialization

The PCS (Prompt, CoT, Solution) dataset is loaded and prepared for batching. We then instantiate the VQ-VAE model using the specific configuration for the codebook size and model dimensions.

In [ ]:
# 1. Data Loading
from src.dataset import Lazy_VQVAE_Dataset
PER_DEVICE_BATCH_SIZE =  2 #4
ACCUM_STEPS = 8 #4
SAVE_FREQUENCY_BATCHES = 5000 # Increased frequency for safety

raw_dataset = load_dataset("meta-math/MetaMathQA")['train']
subset_dataset = raw_dataset.shuffle(seed=42).select(range(50000))

split_datasets = subset_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = Lazy_VQVAE_Dataset(tokenizer, split_datasets['train'], max_length=MAX_SEQ_LEN)
val_dataset = Lazy_VQVAE_Dataset(tokenizer, split_datasets['test'], max_length=MAX_SEQ_LEN)

dataloader = DataLoader(train_dataset, batch_size=PER_DEVICE_BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=PER_DEVICE_BATCH_SIZE, shuffle=False, num_workers=2)

## 2.4 Training Loop

The training process begins here. The model learns to reconstruct mathematical text sequences by mapping them into a discrete latent space. We track the reconstruction loss and the VQ loss to ensure the codebook is being utilized efficiently.

In [ ]:
model = VQVAEModel(
    vocab_size=vocab_size,
    d_model=256,
    num_embeddings=VQ_CODEBOOK_SIZE,
    max_seq_len=MAX_SEQ_LEN, # Ora è 1024
    commitment_cost=0.1
).to(device)

optimizer = optim.Adam(model.parameters(), lr=LR)
criterion = torch.nn.CrossEntropyLoss()

In [ ]:
from tqdm.auto import tqdm

# --- Configuration ---
# Set how often to save within an epoch (e.g., every 1000 batches)
SAVE_STEP_INTERVAL = 1000
start_epoch = 0
start_step = 0  # New tracker for mid-epoch progress
train_losses = []

# --- 1. Logic to Resume Checkpoint ---
# Looks for the most recent file based on epoch and step count
checkpoint_files = [f for f in os.listdir(CHECKPOINT_DIR) if f.endswith('.pth')]

if checkpoint_files:
    # Sorting logic to handle both 'epoch_X.pth' and 'epoch_X_step_Y.pth'
    def sort_key(f):
        parts = f.replace('.pth', '').split('_')
        # Returns (epoch_number, step_number) for accurate sorting
        epoch_val = int(parts[parts.index('epoch') + 1])
        step_val = int(parts[parts.index('step') + 1]) if 'step' in parts else 0
        return (epoch_val, step_val)

    checkpoint_files.sort(key=sort_key)
    latest_checkpoint = os.path.join(CHECKPOINT_DIR, checkpoint_files[-1])

    print(f"🔄 Loading checkpoint: {latest_checkpoint}")
    checkpoint = torch.load(latest_checkpoint, map_location=device)

    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    start_epoch = checkpoint['epoch']
    start_step = checkpoint.get('step', 0) # Get step if it exists, else 0

    print(f"▶️ Starting from epoch {start_epoch + 1}, step {start_step}")
else:
    print("🚀 No checkpoint founded. Starting from scratch...")

# --- 2. Updated Training Loop ---
for epoch in range(start_epoch, EPOCHS):
    model.train()
    total_epoch_loss = 0

    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=True)

    for i, batch in enumerate(progress_bar):
        # Skip batches already processed if we are resuming mid-epoch
        if epoch == start_epoch and i < start_step:
            continue

        input_ids = batch['input_ids'].to(device)

        optimizer.zero_grad()
        loss, recon_loss, vq_loss = model(input_ids)
        loss.backward()
        optimizer.step()

        loss_val = loss.item()
        total_epoch_loss += loss_val
        current_step = i + 1

        progress_bar.set_postfix({
            "loss": f"{loss_val:.4f}",
            "step": current_step
        })

        # --- 3. Mid-Epoch Checkpoint Logic ---
        if current_step % SAVE_STEP_INTERVAL == 0:
            mid_checkpoint_path = os.path.join(
                CHECKPOINT_DIR,
                f"vqvae_epoch_{epoch+1}_step_{current_step}.pth"
            )
            torch.save({
                'epoch': epoch,      # Current epoch (not +1 yet)
                'step': current_step, # Current step
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': loss_val,
            }, mid_checkpoint_path)


    # Reset start_step for the next full epoch
    start_step = 0

    avg_loss = total_epoch_loss / len(dataloader)
    train_losses.append(avg_loss)
    print(f"✅ Epoch {epoch+1} completed. Mean Loss: {avg_loss:.4f}")

    # --- 4. End-of-Epoch Checkpoint ---
    checkpoint_path = os.path.join(CHECKPOINT_DIR, f"vqvae_epoch_{epoch+1}_step_0.pth")
    torch.save({
        'epoch': epoch + 1,
        'step': 0,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': avg_loss,
    }, checkpoint_path)

# Final Save
FINAL_MODEL_PATH = os.path.join(RESULTS_BASE_PATH, "vqvae_final.pth")
torch.save(model.state_dict(), FINAL_MODEL_PATH)
print(f"🏁 Training completed.")